# EXP-013 dev-small 확인: 질의 재작성(H8)

EXP-013 단계 1(`docs/experiments/EXP-013.md`). 재작성 대상 비율, 재작성 출력 형식, 결합 뒤 순위 동작, Google 경로 10건의 지연·형식, 재작성 질의 임베딩 시간, 신호 C 후보 비율을 본다. 판정에 쓰지 않고, 판단 신호·임계값·지시의 뜻은 이 결과로 바꾸지 않는다.

In [1]:
import json
import time
from pathlib import Path

import pandas as pd

from rag.eval.metrics import gold_ranks, load_gold, read_run
from rag.index import load_cfg
from rag.retrieve import Retriever
from rag.rewrite import Rewriter, needs_rewrite, parse, signals
from rag.search import embed_queries, query_embeddings, query_files, query_list, read_texts

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / 'data' / 'runs'
BASE_RUN = 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256-dev-small.csv'
RW_RUN = 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256-rw-dev-small.csv'
REWRITES = ROOT / 'data' / 'rewrites' / 'openrouter-rw_v1.json'
N_GOOGLE = 10

## 재작성 대상 비율

In [3]:
cfg = load_cfg(['+exp=exp013', 'index.scope=dev-small'])
gate = cfg.retriever.rewrite.gate
ret = Retriever(cfg)
qids, text_path = query_list(cfg)
texts = read_texts(text_path, qids)
q_emb = query_embeddings(cfg, qids, text_path, query_files(cfg)['emb'])
tokens = ret.query_tokens(texts)
sig = []
for j, q in enumerate(qids):
    row = ret.chunk_emb @ q_emb[j]
    named, _, _ = ret.lists(row, tokens[j], {})
    sig.append({'qid': q, **signals(named, row, ret.chunk_doc, len(ret.doc_names), gate)})
sig = pd.DataFrame(sig).set_index('qid')
sig['gated'] = [needs_rewrite(s, gate) for s in sig.to_dict('records')]
print(len(sig), 'queries, 재작성 대상', int(sig['gated'].sum()), f"({sig['gated'].mean():.1%})")
print('A′=0', f"{(sig['overlap'] == 0).mean():.1%}",
      ' B′<=임계', f"{(sig['spread'] <= gate.max_spread).mean():.1%}")
print('B′ 분위', sig['spread'].quantile([0.1, 0.25, 0.5]).round(4).to_dict())

1039 queries, 재작성 대상 20 (1.9%)
A′=0 1.0%  B′<=임계 1.3%
B′ 분위 {0.1: 0.124, 0.25: 0.1867, 0.5: 0.2506}


dev-small 인덱스는 문서가 dev-full의 일부라 점수 분포가 다르다. 임계값 0.0588은 dev-full 분포의 하위 10%라서 dev-small의 대상 비율은 dev-full 예상(약 12%)과 다르다.

## 재작성 출력 형식 (OpenRouter, 사고 과정 끔)

In [4]:
saved = json.loads(REWRITES.read_text(encoding='utf-8'))
items = {q: saved['items'][q] for q in sig.index[sig['gated']]}
query_of = dict(zip(qids, texts, strict=True))
fmt = pd.DataFrame([{
    'qid': q, 'seconds': v['seconds'],
    'completion_tokens': v['usage']['completion_tokens'],
    'reasoning_tokens': v['usage']['completion_tokens_details']['reasoning_tokens'],
    'thought_tag': '<thought' in v['raw'], 'lines': len(v['raw'].strip().splitlines()),
    'chars': len(v['raw'].strip()),
    'parsed': parse(v['raw'], cfg.retriever.rewrite.max_chars) is not None,
    'hangul_ratio': (sum('가' <= ch <= '힣' for ch in v['raw'])
                     / max(1, len(v['raw'].replace(' ', '')))),
} for q, v in items.items()]).set_index('qid')
print(fmt.describe().round(3).to_string())
print('형식 통과', int(fmt['parsed'].sum()), '/', len(fmt),
      ' 사고 과정 태그', int(fmt['thought_tag'].sum()), ' 여러 줄', int((fmt['lines'] > 1).sum()))

       seconds  completion_tokens  reasoning_tokens  lines  chars  hangul_ratio
count   20.000              20.00              20.0   20.0  20.00        20.000
mean     1.631              16.70               0.0    1.0  27.40         0.946
std      0.737               5.43               0.0    0.0   8.21         0.111
min      0.471              11.00               0.0    1.0  16.00         0.543
25%      0.922              12.00               0.0    1.0  22.00         0.921
50%      2.093              15.00               0.0    1.0  25.00         1.000
75%      2.189              21.25               0.0    1.0  30.75         1.000
max      2.873              30.00               0.0    1.0  49.00         1.000
형식 통과 20 / 20  사고 과정 태그 0  여러 줄 0


In [5]:
for q in items:
    print('-', query_of[q].strip())
    print('  →', parse(items[q]['raw'], cfg.retriever.rewrite.max_chars))

- 현재 정부의 경제 정책이 경제 회복에 어떠한 역할을 하고 있다고 보십니까?
  → 정부 경제정책 경제 회복 효과 및 역할 분석
- 워킹 홀리데이 퍼센티지를 늘릴 계획이 있나요?
  → 워킹홀리데이(Working Holiday) 비자 쿼터 확대 및 비율 증원 계획
- 다른 주제와 관련하여 용인할 수 있는 대안은 무엇이며, 다른 위원들과 협력하여 진행될 예정인가요?
  → 대안 마련 및 위원 간 협력 방안 추진 계획
- 인건비 부족 문제에 대처하기 위해 월급을 줄이는 방안을 검토할 생각이 있으신가요?
  → 인건비 부족 대응 방안 임금 삭감(월급 줄이는 방안) 검토 여부
- 이 예산이 충분하다고 보는 이유는 무엇인가요?
  → 예산 편성 적정성 및 충분성 사유
- 그러니까 덜컥덜컥 그냥 약속하고 와서 ‘예산편성해 주세요’ 이렇게 하면 이제 안 됩니다. 이게 과거의 일이고. 이게 2020년까지 약속을 한 것이지요?
  → 예산 편성 약속 및 절차 부적절성 2020년 예산 요구
- 10억 가량의 추가 예산은 어떤 목적으로 사용되는지 구체적으로 설명해 주실 수 있을까요?
  → 10억 원 규모 추가 예산 편성 목적 및 사용처 구체적 사유
- 해당 계획을 공개적으로 언급하기 어렵다는데, 이를 추진할 수 있는 여부는 어떤지요?
  → 계획 공개 제한 추진 가능 여부 및 검토
- 다만 기술적인 문제라든가 디테일한 것 때문에 부대의견으로 이걸 하는 것이 훨씬 더 기속력이 있고 확실하다 이렇게 말씀을 하시는 거지요?
  → 기술적 문제 및 세부 사항 부대의견 기속력 실효성
- 분을 법률안에서 시행령으로 변경한 이유는 무엇인가요?
  → 법률안 내용을 시행령으로 변경한 사유 및 근거
- 피해구제에 대한 예산은 정부 예산에서 충분히 지원할 것으로 보이나요? 국회 예산심사 과정에서 다시 검토할 계획이 있나요?
  → 피해구제 예산 정부 지원 충분성 및 국회 예산심사 재검토 계획
- 정부가 경제에 미치는 영향에 대한 우려를 표현했습니다. 어떤 방향으로 정책을 전환해야 한다고 생각하

## 결합 뒤 순위 동작

In [6]:
base, rw = read_run(RUNS / BASE_RUN), read_run(RUNS / RW_RUN)
same = {q: base[base['qid'] == q]['doc_id'].tolist() == rw[rw['qid'] == q]['doc_id'].tolist()
        for q in qids}
changed = {q for q, s in same.items() if not s}
print('순위가 바뀐 질의', len(changed), ' 그중 재작성 대상 밖', len(changed - set(items)))

순위가 바뀐 질의 20  그중 재작성 대상 밖 0


In [7]:
gold = load_gold(cfg, 'dev-small')
ranks = {'EXP-011': gold_ranks(base, gold, 10), 'EXP-013': gold_ranks(rw, gold, 10)}
print(pd.DataFrame({n: {f'recall@{k}': (r <= k).mean() for k in (1, 5, 10)}
                    for n, r in ranks.items()}).round(4))
b, c = ranks['EXP-011'], ranks['EXP-013']
sub = list(items)
print('재작성 대상', len(sub), '건의 정답 순위 (기준 → 재작성)')
print(pd.DataFrame({'base': b[sub], 'rw': c[sub]}).to_string())

           EXP-011  EXP-013
recall@1    0.9259   0.9240
recall@5    0.9865   0.9836
recall@10   0.9894   0.9875
재작성 대상 20 건의 정답 순위 (기준 → 재작성)
             base    rw
qid                    
038711-0001   1.0   1.0
039073-0001   1.0   1.0
043711-0001   1.0  10.0
047529-0007   1.0   1.0
047529-0011   NaN   NaN
047642-0001   1.0   1.0
048643-0007   5.0   NaN
048644-0001   1.0   NaN
048644-0011   1.0   1.0
048689-0002   1.0   1.0
048689-0016   NaN   NaN
049211-0010   6.0   6.0
049233-0004   1.0   2.0
049590-0015   3.0   1.0
050309-0017   NaN   NaN
050951-0003   1.0   1.0
051293-0007   2.0   2.0
052160-0016   1.0   1.0
052415-0012  10.0   NaN
052620-0007   NaN   6.0


## Google 경로 (배포용, 사고 과정을 끌 수 없음)

In [9]:
g_cfg = load_cfg(['+exp=exp013', 'index.scope=dev-small', 'rewriter=gemma_google'])
g_rw = Rewriter(g_cfg)
g_rows = []
for q in list(items)[:N_GOOGLE]:
    out = g_rw.call(query_of[q])
    g_rows.append({'qid': q, 'seconds': out['seconds'],
                   'completion_tokens': out['usage']['completion_tokens'],
                   'thought_tag': '<thought' in out['raw'],
                   'google': parse(out['raw'], cfg.retriever.rewrite.max_chars),
                   'openrouter': parse(items[q]['raw'], cfg.retriever.rewrite.max_chars)})
g = pd.DataFrame(g_rows).set_index('qid')
print(g[['seconds', 'completion_tokens']].describe().round(2).to_string())
print('사고 과정 태그', int(g['thought_tag'].sum()), '/', len(g),
      ' 형식 통과', int(g['google'].notna().sum()),
      ' OpenRouter와 같은 출력', int((g['google'] == g['openrouter']).sum()))

       seconds  completion_tokens
count    10.00              10.00
mean     45.98              17.10
std      43.49               5.13
min      10.20              11.00
25%      14.08              13.25
50%      31.46              15.50
75%      51.67              21.25
max     135.62              25.00
사고 과정 태그 10 / 10  형식 통과 10  OpenRouter와 같은 출력 0


In [10]:
for _, r in g.iterrows():
    print('-', r['openrouter'])
    print('  G', r['google'])

- 정부 경제정책 경제 회복 효과 및 역할 분석
  G 정부 경제 정책 경제 회복 역할 및 효과 영향
- 워킹홀리데이(Working Holiday) 비자 쿼터 확대 및 비율 증원 계획
  G 워킹홀리데이 쿼터(퍼센티지) 확대 및 증원 계획
- 대안 마련 및 위원 간 협력 방안 추진 계획
  G 다른 주제 관련 수용 가능한 대안 및 위원 간 협력 추진 방안
- 인건비 부족 대응 방안 임금 삭감(월급 줄이는 방안) 검토 여부
  G 인건비 부족 대처 임금 삭감(월급을 줄이는 방안) 검토 및 대책
- 예산 편성 적정성 및 충분성 사유
  G 예산 적정성 및 충분성 산출 근거 사유
- 예산 편성 약속 및 절차 부적절성 2020년 예산 요구
  G 2020년까지 약속(합의)에 따른 예산 편성 요구 및 적절성
- 10억 원 규모 추가 예산 편성 목적 및 사용처 구체적 사유
  G 10억 가량 추가 예산 사용 목적 및 집행 계획
- 계획 공개 제한 추진 가능 여부 및 검토
  G 해당 계획 공개 제한 추진 가능 여부 및 실행 가능성
- 기술적 문제 및 세부 사항 부대의견 기속력 실효성
  G 기술적 문제 세부 사항(디테일) 부대의견 기속력 실효성 채택 사유
- 법률안 내용을 시행령으로 변경한 사유 및 근거
  G 법률안 시행령 변경 사유 위임 근거


## 재작성 질의 임베딩 시간 (로컬 CPU)

In [11]:
rw_texts = [parse(v['raw'], cfg.retriever.rewrite.max_chars) for v in items.values()]
start = time.perf_counter()
emb = embed_queries(cfg, rw_texts)
print(len(rw_texts), '건', round(time.perf_counter() - start, 1), '초 (모델 로드 포함)', emb.shape)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1708.99it/s]


20 건 35.8 초 (모델 로드 포함) (20, 1024)


## 신호 C 후보: 조건이 문서를 좁히지 못함 (dev-full, 라벨 미사용)

C의 정의 후보로 두 가지를 센다. (a) 조건 목록이 없음(맞은 필드 수가 `min_fields`에 못 미침), (b) 최다 일치 문서가 많음(100개 초과). 판단 신호에 더하면 재작성 대상이 얼마나 늘어나는지 본다.

In [12]:
from rag.conditions import build_extractor

full_rows = {}
f_ret = None
for qs in ['summary_q', 'multidoc']:
    f_cfg = load_cfg(['+exp=exp013', 'index.scope=full', f'search.queries={qs}'])
    f_ret = f_ret or Retriever(f_cfg)
    f_qids, f_path = query_list(f_cfg)
    f_texts = read_texts(f_path, f_qids)
    f_emb = query_embeddings(f_cfg, f_qids, f_path, query_files(f_cfg)['emb'])
    f_tok = f_ret.query_tokens(f_texts)
    ex = build_extractor(f_cfg)
    rows = []
    for j, q in enumerate(f_qids):
        row = f_ret.chunk_emb @ f_emb[j]
        named, matched, _ = f_ret.lists(row, f_tok[j], ex.extract(f_texts[j]))
        s = signals(named, row, f_ret.chunk_doc, len(f_ret.doc_names), gate)
        rows.append({'qid': q, 'gated': needs_rewrite(s, gate),
                     'no_cond_list': 'conditions' not in named, 'n_best': len(matched)})
    full_rows[qs] = pd.DataFrame(rows).set_index('qid')
    print(qs, len(rows))

summary_q 3016
multidoc 600


In [13]:
for qs, d in full_rows.items():
    wide = d['no_cond_list'] | (d['n_best'] > 100)
    print(qs, {'A′∪B′': f"{d['gated'].mean():.1%}",
               '(a) 조건 목록 없음': f"{d['no_cond_list'].mean():.1%}",
               '(b) 최다 일치 >100': f"{(d['n_best'] > 100).mean():.1%}",
               'A′∪B′∪(a)': f"{(d['gated'] | d['no_cond_list']).mean():.1%}",
               'A′∪B′∪(a∪b)': f"{(d['gated'] | wide).mean():.1%}"})

summary_q {'A′∪B′': '12.0%', '(a) 조건 목록 없음': '99.9%', '(b) 최다 일치 >100': '0.0%', 'A′∪B′∪(a)': '99.9%', 'A′∪B′∪(a∪b)': '99.9%'}
multidoc {'A′∪B′': '48.2%', '(a) 조건 목록 없음': '0.0%', '(b) 최다 일치 >100': '0.0%', 'A′∪B′∪(a)': '48.2%', 'A′∪B′∪(a∪b)': '48.2%'}


## Google 경로 지연 원인 (LangSmith 추적)

위 Google 10건은 평균 46초가 걸렸다. 재작성기에 LangSmith 추적을 붙인 뒤 같은 10건을 다시 호출해, 응답 원문의 사고 과정 길이와 지연의 관계를 본다. 사고 과정이 길수록 오래 걸리면 생성 시간이, 길이와 무관하면 무료 쿼터 대기가 주원인이다. 호출마다 LangSmith에 입력·출력·토큰·지연이 남는다(run 이름 `rewrite`, 메타데이터 `qid`·`rewriter`).

In [14]:
import importlib

import rag.rewrite

importlib.reload(rag.rewrite)
g_rw2 = rag.rewrite.Rewriter(g_cfg)
t_rows = []
for q in list(items)[:N_GOOGLE]:
    out = g_rw2.call(query_of[q], q)
    thought, answer = rag.rewrite.split_thought(out['raw'])
    t_rows.append({'qid': q, 'seconds': out['seconds'], 'thought_chars': len(thought),
                   'answer_chars': len(answer), 'usage': out['usage']})
rag.rewrite.get_cached_client().flush()
t = pd.DataFrame(t_rows).set_index('qid')
print(t[['seconds', 'thought_chars', 'answer_chars']].to_string())
print('spearman(seconds, thought_chars)',
      round(t['seconds'].corr(t['thought_chars'], method='spearman'), 3))
print('초당 사고 과정 글자', (t['thought_chars'] / t['seconds']).describe().round(1).to_dict())
print('usage 예', t['usage'].iloc[0])

InternalServerError: Error code: 500 - [{'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}]

위 셀은 두 번째 호출에서 다시 500을 받아 멈췄다(첫 시도 500에 이어 두 번째, 상태 페이지는 정상). 오류 처리 규칙에 따라 더 시도하지 않았다. LangSmith 기록(run `rewrite`): 성공 1건은 전체 35.6초 중 LLM 호출 35.0초, 사고 과정 1,086자. 실패 1건은 LLM 호출 30.8초(SDK 재시도 2회 포함) 뒤 500. 지연은 모두 API 호출 안에서 생겼고, 대기와 생성의 몫은 표본이 부족해 나누지 못했다.